# Lab 6: Fourier Methods for Potential Fields

| | |
|---|---|
| **Module** | M6 — Fourier Methods, Transformations, and Data Enhancement |
| **Estimated time** | ~2.5 hours |
| **Prerequisites** | Lectures M6-1, M6-2; Homework 6, Problems 6.1–6.2; Lab 3 (total-field anomaly) |
| **Textbook** | Blakely Ch. 11, 12 |

---

## Learning Outcomes

By the end of this lab you will be able to:

1. **Apply** the 2D FFT to a gridded potential field and interpret the amplitude spectrum
2. **Implement** 2D upward continuation in the wavenumber domain and identify its effect on anomaly resolution
3. **Design** a reduction-to-pole (RTP) filter and explain why it fails near the magnetic equator
4. **Estimate** equivalent source depth from the slope of the radially averaged power spectrum

---

## How to use this notebook

- **`[PROVIDED]`** — run as-is
- **`[IMPLEMENT]`** — replace `raise NotImplementedError` with correct code
- **`[VALIDATE]`** — run to check; prints ✓ PASS or ✗ FAIL
- **`[EXPLORE]`** — starting point; modify freely

Hints: `print(hints['key'])`.


## Background

The Fourier transform is the natural language of potential field processing
because Laplace's equation becomes algebraic in the wavenumber domain.  For
a 2D potential field $F(x,y)$, its Fourier transform is

$$
\tilde{F}(k_x, k_y) = \int\int F(x,y)\,e^{-2\pi i(k_x x + k_y y)}\,dx\,dy
$$

where $k_x$, $k_y$ are spatial frequencies (cycles/m) and
$k = \sqrt{k_x^2 + k_y^2}$ is the radial wavenumber.

**Upward continuation** by height $h > 0$ is multiplication by
$e^{-2\pi k h}$ in the wavenumber domain (Blakely §12.1).  This is a
low-pass filter: small $k$ (long wavelengths) are preserved; large $k$
(short wavelengths) are suppressed.

**Reduction to the pole (RTP)** converts the total-field magnetic anomaly to what
it would look like if measured at the magnetic pole ($I = 90°$, $D = 0°$), so
anomaly peaks align with source locations.  The RTP filter in the wavenumber
domain is (Blakely §12.3):

$$
W_{RTP}(k_x, k_y) = \frac{1}{\Theta_m\,\Theta_f}
$$

where $\Theta_m$ and $\Theta_f$ are complex factors that depend on the
magnetization direction and field direction.  For induced magnetization,
$\Theta_m = \Theta_f$, and the filter becomes $1/\Theta_f^2$.

The **radially averaged power spectrum** has the property that for a source
ensemble at depth $z_0$, $\ln|\tilde{F}(k)| \propto -2\pi k z_0$ (Blakely §11.4).
Fitting the slope of the log-power spectrum therefore estimates the equivalent
source depth.


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': False, 'font.size': 11})

def _check(label, got, expected, rtol=1e-4, atol=0.0):
    if np.allclose(got, expected, rtol=rtol, atol=atol):
        print(f'  ✓ PASS  {label}')
    else:
        print(f'  ✗ FAIL  {label}')
        print(f'    expected: {np.asarray(expected)}')
        print(f'    got:      {np.asarray(got)}')

print('Setup complete.')

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
hints = {
    'p1_fft2d': (
        "Use np.fft.fft2 for the 2D transform and np.fft.fftshift to center\n"
        "zero-frequency.  Wavenumber arrays:\n"
        "  kx = np.fft.fftfreq(Nx, d=dx)   [cycles/m, unshifted]\n"
        "  ky = np.fft.fftfreq(Ny, d=dy)\n"
        "  KX, KY = np.meshgrid(kx, ky)   -- note Ny rows, Nx cols\n"
        "  K = sqrt(KX^2 + KY^2)          -- radial wavenumber\n"
        "Apply fftshift to both the spectrum and K before plotting."
    ),
    'p2_upward': (
        "Upward continuation by height h (meters):\n"
        "  1. Compute F_tilde = fft2(data)\n"
        "  2. K = sqrt(KX^2 + KY^2)  [radial wavenumber in cycles/m]\n"
        "  3. Filter: F_tilde_cont = F_tilde * exp(-2*pi*K*h)\n"
        "  4. Result: ifft2(F_tilde_cont).real\n"
        "Note: K is NOT shifted when multiplying — use fftfreq (unshifted)."
    ),
    'p3_rtp_filter': (
        "For induced magnetization (magnetization = field direction) and 2D case:\n"
        "  I = inclination, D = declination (both in radians)\n"
        "  Theta_f = sin(I) + i*cos(I)*(kx*cos(D) + ky*sin(D)) / K\n"
        "         [see Blakely eq. 12.20; be careful with D convention]\n"
        "  W_RTP = 1 / Theta_f^2  (for induced mag)\n"
        "  Set W_RTP = 0 where K = 0 (DC term) to avoid division by zero.\n"
        "  The filter blows up when Theta_f -> 0, i.e., near the equator."
    ),
    'p3_rtp_instability': (
        "Near the magnetic equator (I small), the denominator Theta_f -> 0\n"
        "for wavenumbers perpendicular to the field (kx direction).\n"
        "This means the RTP filter amplifies short-wavelength noise in that\n"
        "direction -> 'ringing' artifacts oriented N-S."
    ),
    'p_spectrum_depth': (
        "Radially average the log power spectrum: for each annulus of K,\n"
        "average log|F_tilde|^2 over all directions.\n"
        "Then fit log(P) vs K with a straight line: slope = -4*pi*z_eq\n"
        "(the 4*pi comes from 2*pi*K = angular wavenumber; check Blakely convention).\n"
        "Use np.polyfit on the linear portion of the spectrum."
    ),
}
# print(hints['p2_upward'])

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Synthetic 2D magnetic anomaly grid.
#
# Three buried dipolar bodies at different depths and locations.
# Regional field: I = 65°, D = 0°.

mu0 = 4 * np.pi * 1e-7

# Grid
dx = dy = 500.0   # m, grid spacing
Nx = Ny = 128
x1d = np.arange(Nx) * dx
y1d = np.arange(Ny) * dy
X, Y = np.meshgrid(x1d, y1d)   # shape (Ny, Nx)
x0 = x1d.mean()
y0 = y1d.mean()

I_reg = np.radians(65.0)   # regional inclination
D_reg = 0.0                 # regional declination

def _dipole_dF_2d(X, Y, xc, yc, depth, m_moment, I_field):
    """Total-field anomaly (nT) from a buried dipole in 3D geometry."""
    Ix = np.cos(I_field)   # northward (x-direction)
    Iz = -np.sin(I_field)  # upward  (z-up convention)
    rx = X - xc
    ry = Y - yc
    rz = depth   # upward distance = depth (source is below)
    r2 = rx**2 + ry**2 + rz**2
    r  = np.sqrt(r2)
    mr = Ix*rx + Iz*rz
    pref = (mu0/(4*np.pi)) * m_moment / r**5
    dBx = pref*(3*mr*rx - Ix*r2)
    dBz = pref*(3*mr*rz - Iz*r2)
    dF = dBx*np.cos(I_field) + dBz*(-np.sin(I_field))
    return dF * 1e9

# Three sources
rng = np.random.default_rng(55)
F_grid  = _dipole_dF_2d(X, Y, x0 - 8e3, y0 + 5e3, 1500.0, 3e12, I_reg)
F_grid += _dipole_dF_2d(X, Y, x0 + 6e3, y0 - 4e3, 4000.0, 8e12, I_reg)
F_grid += _dipole_dF_2d(X, Y, x0 + 2e3, y0 + 8e3, 2500.0, 5e12, I_reg)
F_grid += rng.normal(0, 1.0, size=F_grid.shape)   # 1 nT noise

print(f'Grid: {Nx} × {Ny}, spacing {dx} m')
print(f'Anomaly range: {F_grid.min():.1f} to {F_grid.max():.1f} nT')

---
## Part 1: The 2D Amplitude Spectrum *(Guided — ~35 min)*

Before filtering we need to understand the data in the frequency domain.
We compute the 2D amplitude spectrum, examine its structure, and estimate
a characteristic source depth from the power spectrum slope.

**Available hints:** `p1_fft2d`, `p_spectrum_depth`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Compute the 2D Fourier transform and wavenumber arrays.

def compute_fft2(data, dx, dy):
    """
    Compute 2D FFT and associated wavenumber grids.

    Parameters
    ----------
    data : ndarray, shape (Ny, Nx)
        Gridded field values.
    dx, dy : float
        Grid spacing in x and y directions, meters.

    Returns
    -------
    F_tilde : ndarray, shape (Ny, Nx), complex
        2D Fourier transform of data (NOT shifted).
    KX : ndarray, shape (Ny, Nx)
        Wavenumber in x, cycles/m (NOT shifted).
    KY : ndarray, shape (Ny, Nx)
        Wavenumber in y, cycles/m (NOT shifted).
    K : ndarray, shape (Ny, Nx)
        Radial wavenumber sqrt(KX^2+KY^2), cycles/m (NOT shifted).

    Notes
    -----
    Use np.fft.fft2 and np.fft.fftfreq.
    KX varies along axis=1 (columns); KY varies along axis=0 (rows).
    Return the UNSHIFTED arrays — the filter functions use these directly.
    """
    Ny, Nx = data.shape

    # Step 1: 2D FFT (unshifted)
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 2: wavenumber arrays
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 3: radial wavenumber
    # YOUR CODE HERE
    raise NotImplementedError

    return F_tilde, KX, KY, K

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

F_tilde, KX, KY, K = compute_fft2(F_grid, dx, dy)

# Test 1: Parseval's theorem — energy in space vs. frequency domain
_E_space = np.sum(F_grid**2)
_E_freq  = np.sum(np.abs(F_tilde)**2) / (Nx * Ny)   # normalize
_check('Parseval: E_space ≈ E_freq/N', _E_space, _E_freq, rtol=1e-6)

# Test 2: DC component equals sum of data
_check('DC term = sum(data)', np.abs(F_tilde[0, 0]), abs(F_grid.sum()), rtol=1e-8)

# Test 3: wavenumber shape
_check('KX shape', KX.shape, (Ny, Nx), atol=0)

# Test 4: Nyquist frequency
_kx_nyquist_expected = 1.0 / (2.0 * dx)
_check('Nyquist kx', np.abs(KX).max(), _kx_nyquist_expected, rtol=1e-10)

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Make a 2-panel figure:
#   Left panel: The F_grid anomaly map (nT). Label axes in km.
#   Right panel: The 2D log10 amplitude spectrum (shifted to center DC).
#                Label axes in cycles/km. Use a perceptually uniform colormap.
#
# Add a colorbar to each panel.

# For the spectrum: np.fft.fftshift(np.log10(np.abs(F_tilde) + 1))

# YOUR CODE HERE

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Radially averaged power spectrum and depth estimate.

def radial_power_spectrum(F_tilde, K, dk=None):
    """
    Compute the radially averaged log power spectrum.

    Parameters
    ----------
    F_tilde : ndarray, complex
        2D Fourier transform (unshifted).
    K : ndarray
        Radial wavenumber, cycles/m (unshifted, same shape as F_tilde).
    dk : float, optional
        Bin width in cycles/m.  Default: minimum non-zero K.

    Returns
    -------
    k_centers : ndarray
        Center wavenumber of each bin, cycles/m.
    log_power : ndarray
        Mean log10 power in each bin: log10(mean(|F_tilde|^2)).

    Notes
    -----
    Bin K into annuli of width dk.  Skip the DC bin (K=0).
    Return k_centers and log_power only for bins with at least 4 entries.
    """
    if dk is None:
        dk = K[K > 0].min()

    # Step 1: compute power |F_tilde|^2
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 2: bin by K and average log power
    # YOUR CODE HERE
    raise NotImplementedError

    return k_centers, log_power

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Compute the radial power spectrum and estimate source depth.
#
# 1. Call radial_power_spectrum(F_tilde, K)
# 2. Plot log_power vs k_centers (in cycles/km)
# 3. Fit a straight line to the low-wavenumber portion (k < 0.5 cycles/km)
#    Relationship: log10(P) ≈ const - (4*pi*log10(e)) * z_eq * k
#    => z_eq = -slope / (4*pi*log10(e))
# 4. Overplot the fit and annotate with the estimated depth
# 5. Compare to the range of true source depths (1.5, 2.5, 4.0 km)

# YOUR CODE HERE

### Question 1.1 — Interpreting the spectrum

The power spectrum has a steep roll-off at high wavenumbers.  Is this entirely
due to the depths of the sources, or could other factors contribute?
The spectral depth estimate gives a single equivalent depth — where does it
fall relative to the three true depths (1.5, 2.5, 4.0 km)?  Is this expected?

**Your response:**

> *(Write 4–5 sentences.)*


---
## Part 2: Upward Continuation *(Supported — ~45 min)*

Upward continuation smooths the field and suppresses short-wavelength
noise and shallow sources — useful for separating regional and residual anomalies.

**Available hint:** `p2_upward`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: 2D upward continuation.

def upward_continuation(data, dx, dy, h):
    """
    Upward continue a gridded potential field by height h.

    Parameters
    ----------
    data : ndarray, shape (Ny, Nx)
        Gridded field values.
    dx, dy : float
        Grid spacing, meters.
    h : float
        Continuation height, meters (positive = upward).

    Returns
    -------
    data_cont : ndarray, shape (Ny, Nx)
        Upward-continued field.

    Notes
    -----
    Filter in wavenumber domain: multiply by exp(-2*pi*K*h).
    K must be the UNSHIFTED radial wavenumber (cycles/m).
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# Upward continuation must not increase total power
_F_cont_1 = upward_continuation(F_grid, dx, dy, 500.0)
_F_cont_5 = upward_continuation(F_grid, dx, dy, 5000.0)

if np.std(_F_cont_1) < np.std(F_grid):
    print('  ✓ PASS  Continuation reduces amplitude (h=500 m)')
else:
    print('  ✗ FAIL  Continuation should reduce amplitude')

if np.std(_F_cont_5) < np.std(_F_cont_1):
    print('  ✓ PASS  Higher continuation reduces more (h=5000 > h=500)')
else:
    print('  ✗ FAIL  Higher continuation should reduce more')

# DC value (mean) should be unchanged by upward continuation
_check('Mean preserved by continuation', _F_cont_5.mean(), F_grid.mean(), rtol=1e-5)

# Upward continuing by h=0 should return the original
_F_cont_0 = upward_continuation(F_grid, dx, dy, 0.0)
_check('h=0 returns original', _F_cont_0, F_grid, rtol=1e-6)

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Apply upward continuation at three heights: 500 m, 2000 m, 5000 m.
#
# Make a 2×2 figure:
#   [0,0] Original field
#   [0,1] h = 500 m
#   [1,0] h = 2000 m
#   [1,1] h = 5000 m
#
# Use the same colorscale limits for all panels (vmin/vmax from original).
# Title each panel with the height and the RMS amplitude (np.std).
#
# Also compute the RESIDUAL (original minus h=2000 m continuation)
# and show it in a 5th panel. What does the residual represent geologically?

# YOUR CODE HERE

### Question 2.1 — Geological interpretation

The residual anomaly (original minus upward-continued) isolates which
sources from the three in the synthetic dataset?  Which sources survive
at 5 km continuation?  Explain in terms of the continuation filter and
source depth.

**Your response:**

> *(Write 4–6 sentences.  Reference specific source depths from the dataset.)*


---
## Part 3: Reduction to the Pole *(Open — ~40 min)*

The total-field anomaly has a characteristic asymmetry that depends on
the regional field inclination (as you showed in Lab 3).  Reduction to
the pole (RTP) removes this asymmetry so that anomaly peaks locate
directly above sources.

**Available hints:** `p3_rtp_filter`, `p3_rtp_instability`


### Task 3.1 — Implement and apply the RTP filter

Implement the 2D RTP filter for induced magnetization.  Apply it to `F_grid`
(which was generated at $I = 65°$, $D = 0°$).  Verify that anomaly peaks in
the RTP map align with the known source locations.

Then repeat the RTP for $I = 10°$ (near-equatorial).  Show the result and
explain any artifacts.


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Implement the RTP filter and plot results.
#
# True source locations (for comparison):
source_xy = [
    (x0 - 8e3, y0 + 5e3),   # source 1, depth 1.5 km
    (x0 + 6e3, y0 - 4e3),   # source 2, depth 4.0 km
    (x0 + 2e3, y0 + 8e3),   # source 3, depth 2.5 km
]

# Steps:
# 1. Compute KX, KY, K from compute_fft2
# 2. Compute Theta_f for I=65°, D=0°
# 3. Build RTP filter W_RTP = 1 / Theta_f^2
# 4. Set W_RTP[K==0] = 0
# 5. Apply: F_rtp = ifft2(fft2(F_grid) * W_RTP).real
# 6. Plot F_grid and F_rtp side by side; mark source locations
# 7. Repeat for I=10°

# YOUR CODE HERE

### Question 3.1 — RTP at low inclination

Describe the artifacts that appear in the $I = 10°$ RTP result.
What aspect of the filter formula causes them?  Geophysically, what
would you do if you needed to apply RTP to data collected near the
magnetic equator?

**Your response:**

> *(Write 4–6 sentences.)*


### Question 3.2 — Remanent magnetization

The RTP filter above assumes that magnetization is aligned with the ambient
field (induced magnetization).  If the body has remanent magnetization
in a different direction (as is common in oceanic crust), how would the
RTP result be different?  Could you detect the presence of remanence
from the shape of the anomaly alone?

**Your response:**

> *(Write 4–5 sentences.)*


---
## Synthesis

### S1 — Filter interpretation

You have now seen three wavenumber-domain filters: upward continuation
($e^{-2\pi kh}$), RTP ($1/\Theta_f^2$), and implicitly the derivative operators
($i2\pi k_x$ for the horizontal derivative).  Explain the key difference between
a **smoothing** filter (like upward continuation) and a **phase** filter (like RTP)
in terms of what they do to the amplitude and phase of the spectrum.  Give one
practical advantage and one practical risk for each type.

**Your response:**

> *(Write 4–6 sentences.)*

### S2 — Spectral depth and ambiguity

The spectral depth estimate gave a single equivalent depth for three sources
at different depths.  Under what conditions would the power spectrum clearly
show evidence of two depth populations?  What data requirements (resolution,
spatial extent, noise level) would you need to resolve them?

**Your response:**

> *(Write 4–5 sentences.)*

### S3 — Connection to Lab 6.5

Upward continuation is always stable because the filter $e^{-2\pi kh}$ decays
exponentially for $h > 0$.  **Downward** continuation uses $e^{+2\pi kh}$, which
grows.  Why does this growth make downward continuation numerically dangerous
in a way that upward continuation is not?  What does it imply about what you
can learn from data measured far above the sources?

**Your response:**

> *(Write 4–6 sentences.  This is previewing Lab 6.5.)*


---
## Extensions *(optional)*

### E1 — Analytic signal

The analytic signal amplitude $|A| = \sqrt{(\partial F/\partial x)^2 + (\partial F/\partial y)^2 + (\partial F/\partial z)^2}$
peaks over source edges regardless of magnetization direction.  Implement it
using the Fourier-domain vertical derivative ($|k|\tilde{F}$) and the
horizontal derivatives, and compare peak locations to the source positions.

### E2 — Tilt derivative

The tilt angle $\tau = \arctan(\partial F/\partial z / |\nabla_H F|)$ is
approximately zero over source edges and peaks over source centers.  Implement
it and compare it to the RTP map.  Which gives a sharper localization of sources?

### E3 — Edge tapering

Real survey grids have finite extent, causing edge effects in the FFT.
Apply a cosine taper (20% of the grid width) to `F_grid` before computing the FFT.
How does tapering affect the upward continuation and RTP results?


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Extension workspace.

# YOUR CODE HERE